In [1]:
!pip install librosa soundfile tqdm scikit-learn joblib

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- --------------------


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import subprocess
import json
import math
import warnings
import shutil

import numpy as np
import pandas as pd
import librosa
import joblib

from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [2]:
# Current notebook location
CURRENT_DIR = Path.cwd()

# If your notebook is inside staging/Audio Model or similar,
# adjust PROJECT_ROOT until it points to your project root.
PROJECT_ROOT = CURRENT_DIR

# Example:
# If notebook is in root\Demos\Real Life Trial Data (Demo 5)\train, use: 
PROJECT_ROOT = CURRENT_DIR.parents[2]

DATASET_ROOT = PROJECT_ROOT / "Demos" / "Real Life Trial Data (Demo 5)" / "Dataset" / "Real-life_Deception_Detection_2016"

ANNOTATION_CSV = DATASET_ROOT / "Annotation" / "All_Gestures_Deceptive and Truthful.csv"

DECEPTIVE_VIDEO_DIR = DATASET_ROOT / "Clips" / "Deceptive"
TRUTHFUL_VIDEO_DIR = DATASET_ROOT / "Clips" / "Truthful"

OUTPUT_ROOT = PROJECT_ROOT  / "Demos" / "Real Life Trial Data (Demo 5)" / "train" / "audio_model"
AUDIO_OUTPUT_DIR = OUTPUT_ROOT / "extracted_audio"
PREPARED_OUTPUT_DIR = OUTPUT_ROOT / "prepared_audio_data"

AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Annotation CSV exists:", ANNOTATION_CSV.exists())
print("Deceptive dir exists:", DECEPTIVE_VIDEO_DIR.exists())
print("Truthful dir exists:", TRUTHFUL_VIDEO_DIR.exists())
print("Output root:", OUTPUT_ROOT)

Dataset root: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\Dataset\Real-life_Deception_Detection_2016
Annotation CSV exists: True
Deceptive dir exists: True
Truthful dir exists: True
Output root: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model


In [3]:
# Check FFmpeg locally
ffmpeg_path = shutil.which("ffmpeg")

if ffmpeg_path is None:
    print("FFmpeg was not found in PATH.")
    print("Install FFmpeg, restart VS Code/terminal, and run this cell again.")
else:
    print("FFmpeg found at:", ffmpeg_path)

    result = subprocess.run(
        ["ffmpeg", "-version"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    print(result.stdout.splitlines()[0])

FFmpeg found at: C:\Users\brian\AppData\Local\Microsoft\WinGet\Links\ffmpeg.EXE
ffmpeg version 8.1.1-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers


In [4]:
# Audio processing configuration
SAMPLE_RATE = 16000

WINDOW_SECONDS = 5.0
STRIDE_SECONDS = 2.5

FRAME_LENGTH_SECONDS = 0.064
FRAME_HOP_SECONDS = 0.05

N_MFCC = 13
TARGET_AUDIO_FRAMES = 100

LABEL_DECEPTIVE = 1
LABEL_TRUTHFUL = 0

print("Sample rate:", SAMPLE_RATE)
print("Window seconds:", WINDOW_SECONDS)
print("Stride seconds:", STRIDE_SECONDS)
print("Target audio frames per window:", TARGET_AUDIO_FRAMES)

Sample rate: 16000
Window seconds: 5.0
Stride seconds: 2.5
Target audio frames per window: 100


In [5]:
#Load annotation file
annotations_df = pd.read_csv(ANNOTATION_CSV)

print("Annotation shape:", annotations_df.shape)
print("Columns:", annotations_df.columns.tolist())
annotations_df.head()

Annotation shape: (121, 41)
Columns: ['id', 'OtherGestures', 'Smile', 'Laugh', 'Scowl', 'otherEyebrowMovement', 'Frown', 'Raise', 'OtherEyeMovements', 'Close-R', 'X-Open', 'Close-BE', 'gazeInterlocutor', 'gazeDown', 'gazeUp', 'otherGaze', 'gazeSide', 'openMouth', 'closeMouth', 'lipsDown', 'lipsUp', 'lipsRetracted', 'lipsProtruded', 'SideTurn', 'downR', 'sideTilt', 'backHead', 'otherHeadM', 'sideTurnR', 'sideTiltR', 'waggle', 'forwardHead', 'downRHead', 'singleHand', 'bothHands', 'otherHandM', 'complexHandM', 'sidewaysHand', 'downHands', 'upHands', 'class']


,id,OtherGestures,Smile,Laugh,Scowl,otherEyebrowMovement,Frown,Raise,OtherEyeMovements,Close-R,...,forwardHead,downRHead,singleHand,bothHands,otherHandM,complexHandM,sidewaysHand,downHands,upHands,class
0,trial_lie_001.mp4,1,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,deceptive
1,trial_lie_002.mp4,1,0,0,0,0,1,0,1,0,...,0,0,0,1,0,1,0,0,0,deceptive
2,trial_lie_003.mp4,1,0,0,0,0,1,0,0,1,...,0,0,0,0,1,0,0,0,0,deceptive
3,trial_lie_004.mp4,1,0,0,0,1,0,0,1,0,...,0,1,0,0,1,0,0,0,0,deceptive
4,trial_lie_005.mp4,1,0,0,0,0,1,0,1,0,...,0,0,1,0,0,0,0,0,0,deceptive


In [6]:
#Build video file index
def normalize_name(name):
    name = str(name).strip()
    return Path(name).stem.lower()


def build_video_index():
    video_index = {}

    for video_path in DECEPTIVE_VIDEO_DIR.glob("*.mp4"):
        video_index[normalize_name(video_path.name)] = {
            "video_path": video_path,
            "label": LABEL_DECEPTIVE,
            "label_name": "deceptive",
        }

    for video_path in TRUTHFUL_VIDEO_DIR.glob("*.mp4"):
        video_index[normalize_name(video_path.name)] = {
            "video_path": video_path,
            "label": LABEL_TRUTHFUL,
            "label_name": "truthful",
        }

    return video_index


video_index = build_video_index()

print("Total indexed videos:", len(video_index))
list(video_index.items())[:3]

Total indexed videos: 121


[('trial_lie_001',
  {'video_path': WindowsPath('d:/NSBM/3rd Year/Final Year Project/Product Development/Github/Mutimodal-Deception-Dectection/Demos/Real Life Trial Data (Demo 5)/Dataset/Real-life_Deception_Detection_2016/Clips/Deceptive/trial_lie_001.mp4'),
   'label': 1,
   'label_name': 'deceptive'}),
 ('trial_lie_002',
  {'video_path': WindowsPath('d:/NSBM/3rd Year/Final Year Project/Product Development/Github/Mutimodal-Deception-Dectection/Demos/Real Life Trial Data (Demo 5)/Dataset/Real-life_Deception_Detection_2016/Clips/Deceptive/trial_lie_002.mp4'),
   'label': 1,
   'label_name': 'deceptive'}),
 ('trial_lie_003',
  {'video_path': WindowsPath('d:/NSBM/3rd Year/Final Year Project/Product Development/Github/Mutimodal-Deception-Dectection/Demos/Real Life Trial Data (Demo 5)/Dataset/Real-life_Deception_Detection_2016/Clips/Deceptive/trial_lie_003.mp4'),
   'label': 1,
   'label_name': 'deceptive'})]

In [7]:
#Create training video list
training_videos = []

for _, row in annotations_df.iterrows():
    video_id = normalize_name(row["id"])
    class_name = str(row["class"]).strip().lower()

    if video_id not in video_index:
        continue

    item = video_index[video_id]

    training_videos.append({
        "video_id": video_id,
        "video_path": item["video_path"],
        "label": item["label"],
        "label_name": item["label_name"],
    })

training_videos_df = pd.DataFrame(training_videos)

print("Matched training videos:", len(training_videos_df))
print(training_videos_df["label_name"].value_counts())

training_videos_df.head()

Matched training videos: 121
label_name
deceptive    61
truthful     60
Name: count, dtype: int64


,video_id,video_path,label,label_name
0,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive
1,trial_lie_002,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive
2,trial_lie_003,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive
3,trial_lie_004,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive
4,trial_lie_005,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive


In [8]:
#Extract audio from video using FFmpeg
def extract_audio_to_wav(video_path, output_dir):
    """
    Extracts mono 16 kHz WAV audio from a video file using FFmpeg.

    This requires FFmpeg to be installed locally and available in PATH.
    """

    video_path = Path(video_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_wav_path = output_dir / f"{video_path.stem}.wav"

    if output_wav_path.exists():
        return output_wav_path

    if shutil.which("ffmpeg") is None:
        raise RuntimeError(
            "FFmpeg was not found in PATH. Install FFmpeg and restart VS Code/terminal."
        )

    command = [
        "ffmpeg",
        "-y",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(SAMPLE_RATE),
        str(output_wav_path),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"FFmpeg failed for {video_path.name}\n{result.stderr}"
        )

    return output_wav_path

In [9]:
#Extract all audio files
audio_records = []

for item in tqdm(training_videos, desc="Extracting audio"):
    try:
        wav_path = extract_audio_to_wav(
            item["video_path"],
            AUDIO_OUTPUT_DIR / item["label_name"]
        )

        audio_records.append({
            **item,
            "wav_path": wav_path,
        })

    except Exception as error:
        print("Failed:", item["video_path"], error)

audio_records_df = pd.DataFrame(audio_records)

print("Extracted audio files:", len(audio_records_df))
audio_records_df.head()

Extracting audio: 100%|██████████| 121/121 [00:00<00:00, 8532.60it/s]

Extracted audio files: 121


,video_id,video_path,label,label_name,wav_path
0,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,d:\NSBM\3rd Year\Final Year Project\Product De...
1,trial_lie_002,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,d:\NSBM\3rd Year\Final Year Project\Product De...
2,trial_lie_003,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,d:\NSBM\3rd Year\Final Year Project\Product De...
3,trial_lie_004,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,d:\NSBM\3rd Year\Final Year Project\Product De...
4,trial_lie_005,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,d:\NSBM\3rd Year\Final Year Project\Product De...


In [10]:
#Define audio feature extraction function
# MFCC
# Delta MFCC
# Delta-delta MFCC
# RMS energy
# Zero-crossing rate
# Spectral centroid
# Spectral bandwidth
# Spectral rolloff
# Pitch estimate

def pad_or_trim_frames(feature_matrix, target_frames):
    """
    feature_matrix shape: (time_steps, feature_count)
    """

    current_frames = feature_matrix.shape[0]

    if current_frames == target_frames:
        return feature_matrix

    if current_frames > target_frames:
        return feature_matrix[:target_frames]

    pad_amount = target_frames - current_frames
    padding = np.zeros((pad_amount, feature_matrix.shape[1]), dtype=np.float32)

    return np.vstack([feature_matrix, padding])


def extract_audio_features_from_window(y_window, sr):
    """
    Returns:
        feature_matrix shape: (TARGET_AUDIO_FRAMES, feature_count)
    """

    frame_length = int(FRAME_LENGTH_SECONDS * sr)
    hop_length = int(FRAME_HOP_SECONDS * sr)

    if len(y_window) < frame_length:
        y_window = np.pad(y_window, (0, frame_length - len(y_window)))

    mfcc = librosa.feature.mfcc(
        y=y_window,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=1024,
        hop_length=hop_length
    )

    delta_mfcc = librosa.feature.delta(mfcc)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)

    rms = librosa.feature.rms(
        y=y_window,
        frame_length=frame_length,
        hop_length=hop_length
    )

    zcr = librosa.feature.zero_crossing_rate(
        y_window,
        frame_length=frame_length,
        hop_length=hop_length
    )

    spectral_centroid = librosa.feature.spectral_centroid(
        y=y_window,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length
    )

    spectral_bandwidth = librosa.feature.spectral_bandwidth(
        y=y_window,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length
    )

    spectral_rolloff = librosa.feature.spectral_rolloff(
        y=y_window,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length
    )

    # Pitch estimation using librosa.pyin
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(
            y_window,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr,
            frame_length=1024,
            hop_length=hop_length
        )

        f0 = np.nan_to_num(f0, nan=0.0)
        voiced_probs = np.nan_to_num(voiced_probs, nan=0.0)

    except Exception:
        frame_count = mfcc.shape[1]
        f0 = np.zeros(frame_count)
        voiced_probs = np.zeros(frame_count)

    min_frames = min(
        mfcc.shape[1],
        delta_mfcc.shape[1],
        delta2_mfcc.shape[1],
        rms.shape[1],
        zcr.shape[1],
        spectral_centroid.shape[1],
        spectral_bandwidth.shape[1],
        spectral_rolloff.shape[1],
        len(f0),
        len(voiced_probs)
    )

    feature_stack = np.vstack([
        mfcc[:, :min_frames],
        delta_mfcc[:, :min_frames],
        delta2_mfcc[:, :min_frames],
        rms[:, :min_frames],
        zcr[:, :min_frames],
        spectral_centroid[:, :min_frames],
        spectral_bandwidth[:, :min_frames],
        spectral_rolloff[:, :min_frames],
        f0[:min_frames].reshape(1, -1),
        voiced_probs[:min_frames].reshape(1, -1),
    ])

    feature_matrix = feature_stack.T.astype(np.float32)

    feature_matrix = pad_or_trim_frames(
        feature_matrix,
        TARGET_AUDIO_FRAMES
    )

    return feature_matrix


In [11]:
# Create windows from one audio file
def create_audio_windows_for_file(wav_path, label, label_name, video_id):
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)

    duration = len(y) / sr

    window_samples = int(WINDOW_SECONDS * sr)
    stride_samples = int(STRIDE_SECONDS * sr)

    windows = []
    metadata = []

    if len(y) == 0:
        return windows, metadata

    if len(y) < window_samples:
        padded_y = np.pad(y, (0, window_samples - len(y)))
        feature_matrix = extract_audio_features_from_window(padded_y, sr)

        windows.append(feature_matrix)
        metadata.append({
            "video_id": video_id,
            "wav_path": str(wav_path),
            "label": label,
            "label_name": label_name,
            "start_time": 0.0,
            "end_time": min(WINDOW_SECONDS, duration),
            "duration": duration,
        })

        return windows, metadata

    start_sample = 0

    while start_sample + window_samples <= len(y):
        end_sample = start_sample + window_samples

        y_window = y[start_sample:end_sample]

        feature_matrix = extract_audio_features_from_window(y_window, sr)

        start_time = start_sample / sr
        end_time = end_sample / sr

        windows.append(feature_matrix)
        metadata.append({
            "video_id": video_id,
            "wav_path": str(wav_path),
            "label": label,
            "label_name": label_name,
            "start_time": start_time,
            "end_time": end_time,
            "duration": duration,
        })

        start_sample += stride_samples

    return windows, metadata

In [12]:
# Extract all audio training windows
all_features = []
all_labels = []
all_metadata = []

for _, row in tqdm(audio_records_df.iterrows(), total=len(audio_records_df), desc="Extracting audio features"):
    try:
        windows, metadata = create_audio_windows_for_file(
            wav_path=row["wav_path"],
            label=int(row["label"]),
            label_name=row["label_name"],
            video_id=row["video_id"],
        )

        for feature_matrix, meta in zip(windows, metadata):
            all_features.append(feature_matrix)
            all_labels.append(int(row["label"]))
            all_metadata.append(meta)

    except Exception as error:
        print("Failed feature extraction:", row["video_id"], error)

X_audio = np.array(all_features, dtype=np.float32)
y_audio = np.array(all_labels, dtype=np.int64)
metadata_df = pd.DataFrame(all_metadata)

print("X_audio shape:", X_audio.shape)
print("y_audio shape:", y_audio.shape)
print("Metadata shape:", metadata_df.shape)
print(metadata_df["label_name"].value_counts())
metadata_df.head()

Extracting audio features: 100%|██████████| 121/121 [30:51<00:00, 15.30s/it]

X_audio shape: (1179, 100, 46)
y_audio shape: (1179,)
Metadata shape: (1179, 7)
label_name
truthful     593
deceptive    586
Name: count, dtype: int64


,video_id,wav_path,label,label_name,start_time,end_time,duration
0,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,0.0,5.0,16.96
1,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,2.5,7.5,16.96
2,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,5.0,10.0,16.96
3,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,7.5,12.5,16.96
4,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,10.0,15.0,16.96


In [13]:
# Check for invalid values
print("NaN count:", np.isnan(X_audio).sum())
print("Inf count:", np.isinf(X_audio).sum())

X_audio = np.nan_to_num(X_audio, nan=0.0, posinf=0.0, neginf=0.0)

print("After cleanup")
print("NaN count:", np.isnan(X_audio).sum())
print("Inf count:", np.isinf(X_audio).sum())

NaN count: 0
Inf count: 0
After cleanup
NaN count: 0
Inf count: 0


In [14]:
# Check for invalid values
print("NaN count:", np.isnan(X_audio).sum())
print("Inf count:", np.isinf(X_audio).sum())

X_audio = np.nan_to_num(X_audio, nan=0.0, posinf=0.0, neginf=0.0)

print("After cleanup")
print("NaN count:", np.isnan(X_audio).sum())
print("Inf count:", np.isinf(X_audio).sum())

NaN count: 0
Inf count: 0
After cleanup
NaN count: 0
Inf count: 0


In [15]:
# Scale audio features
num_windows, time_steps, feature_count = X_audio.shape

X_reshaped = X_audio.reshape(-1, feature_count)

scaler = StandardScaler()
X_scaled_reshaped = scaler.fit_transform(X_reshaped)

X_audio_scaled = X_scaled_reshaped.reshape(num_windows, time_steps, feature_count).astype(np.float32)

print("Original shape:", X_audio.shape)
print("Scaled shape:", X_audio_scaled.shape)
print("Feature count:", feature_count)

Original shape: (1179, 100, 46)
Scaled shape: (1179, 100, 46)
Feature count: 46


In [16]:
# Save prepared audio dataset
np.save(PREPARED_OUTPUT_DIR / "audio_features.npy", X_audio_scaled)
np.save(PREPARED_OUTPUT_DIR / "audio_labels.npy", y_audio)

metadata_df.to_csv(PREPARED_OUTPUT_DIR / "audio_window_metadata.csv", index=False)

joblib.dump(scaler, PREPARED_OUTPUT_DIR / "audio_feature_scaler.pkl")

feature_info = {
    "sample_rate": SAMPLE_RATE,
    "window_seconds": WINDOW_SECONDS,
    "stride_seconds": STRIDE_SECONDS,
    "frame_length_seconds": FRAME_LENGTH_SECONDS,
    "frame_hop_seconds": FRAME_HOP_SECONDS,
    "target_audio_frames": TARGET_AUDIO_FRAMES,
    "n_mfcc": N_MFCC,
    "feature_count": int(feature_count),
    "features": [
        "mfcc_1_to_13",
        "delta_mfcc_1_to_13",
        "delta2_mfcc_1_to_13",
        "rms",
        "zero_crossing_rate",
        "spectral_centroid",
        "spectral_bandwidth",
        "spectral_rolloff",
        "pitch_f0",
        "voicing_probability"
    ],
    "label_mapping": {
        "truthful": 0,
        "deceptive": 1
    }
}

with open(PREPARED_OUTPUT_DIR / "feature_info.json", "w") as file:
    json.dump(feature_info, file, indent=4)

print("Saved files:")
for path in PREPARED_OUTPUT_DIR.iterdir():
    print(path)

Saved files:
d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\prepared_audio_data\audio_features.npy
d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\prepared_audio_data\audio_feature_scaler.pkl
d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\prepared_audio_data\audio_labels.npy
d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\prepared_audio_data\audio_window_metadata.csv
d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\prepared_audio_data\feature_info.json


In [17]:
# Quick verification
loaded_X = np.load(PREPARED_OUTPUT_DIR / "audio_features.npy")
loaded_y = np.load(PREPARED_OUTPUT_DIR / "audio_labels.npy")
loaded_meta = pd.read_csv(PREPARED_OUTPUT_DIR / "audio_window_metadata.csv")

print("Loaded X:", loaded_X.shape)
print("Loaded y:", loaded_y.shape)
print("Loaded metadata:", loaded_meta.shape)

print("Labels:")
print(pd.Series(loaded_y).value_counts())

loaded_meta.head()

Loaded X: (1179, 100, 46)
Loaded y: (1179,)
Loaded metadata: (1179, 7)
Labels:
0    593
1    586
Name: count, dtype: int64


,video_id,wav_path,label,label_name,start_time,end_time,duration
0,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,0.0,5.0,16.96
1,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,2.5,7.5,16.96
2,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,5.0,10.0,16.96
3,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,7.5,12.5,16.96
4,trial_lie_001,d:\NSBM\3rd Year\Final Year Project\Product De...,1,deceptive,10.0,15.0,16.96
